# 02 — Structured vs. Semantic Retrieval Demo

Demonstrates Chapter 2's architectural point concretely: a precise structured lookup for synthetic KYC/transaction data vs. TF-IDF-based semantic retrieval over synthetic prior case notes — showing why each source uses the retrieval method it does. Fully offline (scikit-learn TF-IDF stands in for an Azure AI Search embedding index).

In [1]:
# --- Structured data store: exact, deterministic lookups (the KYC / transaction tools) ---
_KYC_DB = {
    'CUST-4471': {'occupation': 'Import/export consultant', 'stated_income': 95000, 'risk_rating': 'MEDIUM'},
    'CUST-5502': {'occupation': 'Retail store owner', 'stated_income': 60000, 'risk_rating': 'LOW'},
}
_TXN_DB = {
    'CUST-4471': [
        {'transaction_id': 'TXN-88201', 'amount': 9800, 'date': '2026-06-01'},
        {'transaction_id': 'TXN-88213', 'amount': 9700, 'date': '2026-06-02'},
        {'transaction_id': 'TXN-88240', 'amount': 9650, 'date': '2026-06-03'},
        {'transaction_id': 'TXN-70011', 'amount': 250, 'date': '2026-01-15'},
    ],
}

def get_kyc_profile(customer_id: str) -> dict:
    """Deterministic lookup -- one row, exact fields, no ranking involved."""
    return _KYC_DB[customer_id]

def get_transaction_history(customer_id: str, start_date: str, end_date: str, min_amount: float = None):
    """Deterministic, filtered query -- an explicit, auditable lookback window."""
    rows = [t for t in _TXN_DB[customer_id] if start_date <= t['date'] <= end_date]
    if min_amount is not None:
        rows = [t for t in rows if t['amount'] >= min_amount]
    return rows

kyc = get_kyc_profile('CUST-4471')
txns = get_transaction_history('CUST-4471', '2026-06-01', '2026-06-30')
print('KYC (exact):', kyc)
print(f"Transactions in window (exact, {len(txns)} rows):")
for t in txns:
    print(' ', t)

KYC (exact): {'occupation': 'Import/export consultant', 'stated_income': 95000, 'risk_rating': 'MEDIUM'}
Transactions in window (exact, 3 rows):
  {'transaction_id': 'TXN-88201', 'amount': 9800, 'date': '2026-06-01'}
  {'transaction_id': 'TXN-88213', 'amount': 9700, 'date': '2026-06-02'}
  {'transaction_id': 'TXN-88240', 'amount': 9650, 'date': '2026-06-03'}


## Semantic retrieval over prior case notes

Case notes are free text — this is the one source where fuzzy relevance ranking (TF-IDF cosine similarity, standing in for Azure AI Search's embedding-based retrieval) is the right tool, because there is no fixed schema to `SELECT` a semantically-similar note from.

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

prior_case_notes = [
    {'case_id': 'CASE-2025-0033', 'customer_id': 'CUST-4471',
     'text': 'Customer flagged for near-threshold wire pattern; closed as false positive after invoice documentation confirmed legitimate trade financing.'},
    {'case_id': 'CASE-2025-0090', 'customer_id': 'CUST-4471',
     'text': 'Elevated inbound wire activity from overseas counterparties reviewed; customer provided commercial contracts supporting the payments.'},
    {'case_id': 'CASE-2024-0187', 'customer_id': 'CUST-5502',
     'text': 'Unrelated retail customer cash-deposit pattern review; no connection to import/export activity.'},
]

query = 'unusual receipt pattern from overseas counterparties, possible structuring'

# Metadata pre-filter first (customer_id), THEN semantic ranking within that filtered set --
# exactly the compose-not-compete pattern Chapter 2 describes.
customer_notes = [n for n in prior_case_notes if n['customer_id'] == 'CUST-4471']
corpus = [n['text'] for n in customer_notes]

vectorizer = TfidfVectorizer(stop_words='english')
doc_vectors = vectorizer.fit_transform(corpus)
query_vector = vectorizer.transform([query])

scores = cosine_similarity(query_vector, doc_vectors)[0]
ranked = sorted(zip(customer_notes, scores), key=lambda x: -x[1])

print(f"Query: {query!r}\n")
print('Ranked (filtered to CUST-4471 only, then semantically ranked):')
for note, score in ranked:
    print(f"  [{score:.3f}] {note['case_id']}: {note['text'][:80]}...")

assert all(n['customer_id'] == 'CUST-4471' for n, _ in ranked), (
    'Structured pre-filter must exclude other customers entirely -- semantic search '
    'never even sees CASE-2024-0187, matching Chapter 2\'s "compose, don\'t compete" design.'
)
print('\nPASS: unrelated customer (CUST-5502) never entered the semantic search at all.')

Query: 'unusual receipt pattern from overseas counterparties, possible structuring'

Ranked (filtered to CUST-4471 only, then semantically ranked):
  [0.333] CASE-2025-0090: Elevated inbound wire activity from overseas counterparties reviewed; customer p...
  [0.154] CASE-2025-0033: Customer flagged for near-threshold wire pattern; closed as false positive after...

PASS: unrelated customer (CUST-5502) never entered the semantic search at all.


## Why 'top-k similar transactions' is the wrong query

Demonstrating Chapter 2's completeness argument directly: a similarity-ranked "top-k transactions" query would silently drop transactions relevant to the alert window, where a deterministic date/amount filter returns the complete, correct set every time.

In [3]:
# A naive 'top-k similar transactions by amount' embedding-style query -- the wrong tool
target_amount = 9700
all_txns = _TXN_DB['CUST-4471']
by_amount_closeness = sorted(all_txns, key=lambda t: abs(t['amount'] - target_amount))[:2]
print('Naive similarity-style top-2 by amount (WRONG for this use case):')
for t in by_amount_closeness:
    print(' ', t)
print('-> misses TXN-88201 entirely, which is part of the same structuring pattern.\n')

# The correct structured query: complete, deterministic, auditable
correct = get_transaction_history('CUST-4471', '2026-06-01', '2026-06-03', min_amount=9000)
print(f"Structured filter (exact, complete, {len(correct)} rows):")
for t in correct:
    print(' ', t)

assert len(correct) == 3, 'The structured query must return the complete windowed set.'
print('\nPASS: the structured query returns the complete, correct set; the similarity-style query does not.')

Naive similarity-style top-2 by amount (WRONG for this use case):
  {'transaction_id': 'TXN-88213', 'amount': 9700, 'date': '2026-06-02'}
  {'transaction_id': 'TXN-88240', 'amount': 9650, 'date': '2026-06-03'}
-> misses TXN-88201 entirely, which is part of the same structuring pattern.

Structured filter (exact, complete, 3 rows):
  {'transaction_id': 'TXN-88201', 'amount': 9800, 'date': '2026-06-01'}
  {'transaction_id': 'TXN-88213', 'amount': 9700, 'date': '2026-06-02'}
  {'transaction_id': 'TXN-88240', 'amount': 9650, 'date': '2026-06-03'}

PASS: the structured query returns the complete, correct set; the similarity-style query does not.
